In [39]:
import os
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [40]:
PROJECT_DIR = os.path.abspath("..")

DATA_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "cases.csv"
)

df = pd.read_csv(DATA_PATH)

df.head()

,case_id,nomor_putusan,pengadilan,tahun,nama_terdakwa,pasal,amar_putusan,isi_putusan
0,case_001,166/Pid.B/2019/PN,Pengadilan Negeri Cibinong yang mengadili perk...,2019,Randy Alias Boy Bin Odih 2. Tempat lahir : Bog...,-,mengadili perkara pidana doengan u acara pemer...,a i s e n o d n I k i l b u p e a R i s g e n ...
1,case_002,182/Pid.B/2021/PN,Pengadilan Negeri Cibinong yang mengadili perk...,2021,Odih Al. Belong Bin Rohaca h k 2. Tempat lahir...,-,mengadili perkara pidana dengan d agcara pemer...,a i s e n o d n I k i l b u p e a R i s g e n ...
2,case_003,182/Pid.B/2021/PN,Pengadilan Negeri Cibinong yang mengadili perk...,2021,Odih Al. Belong Bin Rohaca h k 2. Tempat lahir...,-,mengadili perkara pidana dengan d agcara pemer...,a i s e n o d n I k i l b u p e a R i s g e n ...
3,case_004,tanggal,Pengadilan Negeri Cibinong yang memeriksa dan ...,2020,Inen Bin Boin.Alm; a Tempat lahir : Bogor; i l...,-,mengadili perkara-perkara n Apidana pada perad...,a i s e n o d n I k i l b u p e a R i s g e n ...
4,case_005,190/Pid.B/2023/PN,Pengadilan Negeri Cibinong yang mengadili perk...,2023,Lusky Alias Ulus; 2. Tempat lahir : Karawang; ...,-,mengadili perkara pidana doengan u acara pemer...,a i s e n o d n I k i l b u p e a R i s g e n ...


In [41]:
def clean_text(text):

    text = str(text).lower()

    # hapus karakter aneh
    text = re.sub(r'[^a-zA-Z0-9 ]',' ',text)

    # hapus spasi berlebih
    text = re.sub(r'\s+',' ',text)

    # buang huruf satuan hasil OCR
    text = re.sub(r'\b[a-zA-Z]\b',' ',text)

    return text.strip()

In [42]:
df["clean_text"] = (
    df["isi_putusan"]
    .fillna("")
    .astype(str)
    .apply(clean_text)
)

df.head()

,case_id,nomor_putusan,pengadilan,tahun,nama_terdakwa,pasal,amar_putusan,isi_putusan,clean_text
0,case_001,166/Pid.B/2019/PN,Pengadilan Negeri Cibinong yang mengadili perk...,2019,Randy Alias Boy Bin Odih 2. Tempat lahir : Bog...,-,mengadili perkara pidana doengan u acara pemer...,a i s e n o d n I k i l b u p e a R i s g e n ...,direktori putusan mahkamah agung republik indo...
1,case_002,182/Pid.B/2021/PN,Pengadilan Negeri Cibinong yang mengadili perk...,2021,Odih Al. Belong Bin Rohaca h k 2. Tempat lahir...,-,mengadili perkara pidana dengan d agcara pemer...,a i s e n o d n I k i l b u p e a R i s g e n ...,direktori putusan mahkamah agung republik indo...
2,case_003,182/Pid.B/2021/PN,Pengadilan Negeri Cibinong yang mengadili perk...,2021,Odih Al. Belong Bin Rohaca h k 2. Tempat lahir...,-,mengadili perkara pidana dengan d agcara pemer...,a i s e n o d n I k i l b u p e a R i s g e n ...,direktori putusan mahkamah agung republik indo...
3,case_004,tanggal,Pengadilan Negeri Cibinong yang memeriksa dan ...,2020,Inen Bin Boin.Alm; a Tempat lahir : Bogor; i l...,-,mengadili perkara-perkara n Apidana pada perad...,a i s e n o d n I k i l b u p e a R i s g e n ...,direktori putusan mahkamah agung republik indo...
4,case_005,190/Pid.B/2023/PN,Pengadilan Negeri Cibinong yang mengadili perk...,2023,Lusky Alias Ulus; 2. Tempat lahir : Karawang; ...,-,mengadili perkara pidana doengan u acara pemer...,a i s e n o d n I k i l b u p e a R i s g e n ...,direktori putusan mahkamah agung republik indo...


In [43]:
vectorizer = TfidfVectorizer(
    stop_words=None,
    ngram_range=(1,2),
    max_features=8000
)

tfidf_matrix = vectorizer.fit_transform(
    df["clean_text"]
)

print(tfidf_matrix.shape)

(30, 3138)


In [44]:
query = df.loc[0,"clean_text"][:1500]

In [45]:
query_vector = vectorizer.transform([query])

In [46]:
similarity = cosine_similarity(
    query_vector,
    tfidf_matrix
).flatten()

similarity

array([0.95570262, 0.20128479, 0.20128479, 0.15901227, 0.16029746,
       0.18444118, 0.55402254, 0.16914004, 0.18254567, 0.15171223,
       0.06849631, 0.12867579, 0.14627785, 0.12171331, 0.17874423,
       0.17230753, 0.14922401, 0.14740911, 0.19021525, 0.16982456,
       0.18897035, 0.1735507 , 0.15428012, 0.16993547, 0.14635317,
       0.14635317, 0.15700088, 0.13872419, 0.14239236, 0.21465062])

In [47]:
top_idx = np.argsort(similarity)[::-1][:5]

results = df.iloc[top_idx][[
    "case_id",
    "nomor_putusan",
    "nama_terdakwa",
    "pasal",
    "amar_putusan"
]].copy()

results["similarity"] = similarity[top_idx]

results

,case_id,nomor_putusan,nama_terdakwa,pasal,amar_putusan,similarity
0,case_001,166/Pid.B/2019/PN,Randy Alias Boy Bin Odih 2. Tempat lahir : Bog...,-,mengadili perkara pidana doengan u acara pemer...,0.955703
6,case_007,241/Pid.B/2019/PN,Nano Bin Patma; I Tempat lahir : Bogor; h k Um...,-,mengadili perkara pidana doengan u acara pemer...,0.554023
29,case_030,99/Pid.B/2019/PN,ISAK Bin IDING n A Tempat lahir : Sukabumi I U...,pasal 480 KRe,mengadili perkara pidana dengan o u acara peme...,0.214651
1,case_002,182/Pid.B/2021/PN,Odih Al. Belong Bin Rohaca h k 2. Tempat lahir...,-,mengadili perkara pidana dengan d agcara pemer...,0.201285
2,case_003,182/Pid.B/2021/PN,Odih Al. Belong Bin Rohaca h k 2. Tempat lahir...,-,mengadili perkara pidana dengan d agcara pemer...,0.201285


In [48]:
RESULT_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "results"
)

os.makedirs(RESULT_DIR,exist_ok=True)

results.to_csv(
    os.path.join(
        RESULT_DIR,
        "retrieval_results.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

print("Retrieval berhasil disimpan.")

Retrieval berhasil disimpan.


In [49]:
results

,case_id,nomor_putusan,nama_terdakwa,pasal,amar_putusan,similarity
0,case_001,166/Pid.B/2019/PN,Randy Alias Boy Bin Odih 2. Tempat lahir : Bog...,-,mengadili perkara pidana doengan u acara pemer...,0.955703
6,case_007,241/Pid.B/2019/PN,Nano Bin Patma; I Tempat lahir : Bogor; h k Um...,-,mengadili perkara pidana doengan u acara pemer...,0.554023
29,case_030,99/Pid.B/2019/PN,ISAK Bin IDING n A Tempat lahir : Sukabumi I U...,pasal 480 KRe,mengadili perkara pidana dengan o u acara peme...,0.214651
1,case_002,182/Pid.B/2021/PN,Odih Al. Belong Bin Rohaca h k 2. Tempat lahir...,-,mengadili perkara pidana dengan d agcara pemer...,0.201285
2,case_003,182/Pid.B/2021/PN,Odih Al. Belong Bin Rohaca h k 2. Tempat lahir...,-,mengadili perkara pidana dengan d agcara pemer...,0.201285


In [50]:
all_similarity = df.copy()

all_similarity["similarity"] = similarity

all_similarity.sort_values(
    by="similarity",
    ascending=False,
    inplace=True
)

all_similarity.head(10)

,case_id,nomor_putusan,pengadilan,tahun,nama_terdakwa,pasal,amar_putusan,isi_putusan,clean_text,similarity
0,case_001,166/Pid.B/2019/PN,Pengadilan Negeri Cibinong yang mengadili perk...,2019,Randy Alias Boy Bin Odih 2. Tempat lahir : Bog...,-,mengadili perkara pidana doengan u acara pemer...,a i s e n o d n I k i l b u p e a R i s g e n ...,direktori putusan mahkamah agung republik indo...,0.955703
6,case_007,241/Pid.B/2019/PN,Pengadilan Negeri Cibinong yang mengadili perk...,2019,Nano Bin Patma; I Tempat lahir : Bogor; h k Um...,-,mengadili perkara pidana doengan u acara pemer...,a i s e n o d n I k i l b u p e a R i s g e n ...,direktori putusan mahkamah agung republik indo...,0.554023
29,case_030,99/Pid.B/2019/PN,Pengadilan Negeri Cibinong yang mengadili perk...,2019,ISAK Bin IDING n A Tempat lahir : Sukabumi I U...,pasal 480 KRe,mengadili perkara pidana dengan o u acara peme...,a i s e n o d n I k i l b u p e a R i s g e n ...,direktori putusan mahkamah agung republik indo...,0.214651
2,case_003,182/Pid.B/2021/PN,Pengadilan Negeri Cibinong yang mengadili perk...,2021,Odih Al. Belong Bin Rohaca h k 2. Tempat lahir...,-,mengadili perkara pidana dengan d agcara pemer...,a i s e n o d n I k i l b u p e a R i s g e n ...,direktori putusan mahkamah agung republik indo...,0.201285
1,case_002,182/Pid.B/2021/PN,Pengadilan Negeri Cibinong yang mengadili perk...,2021,Odih Al. Belong Bin Rohaca h k 2. Tempat lahir...,-,mengadili perkara pidana dengan d agcara pemer...,a i s e n o d n I k i l b u p e a R i s g e n ...,direktori putusan mahkamah agung republik indo...,0.201285
18,case_019,438/Pid.B/2022/PN,Pengadilan Negeri Cibinong yang mengadili perk...,2022,Fachmi Agustiani I h2. Tempat lahir : Jakarta ...,-,mengadili perkara pidana dengan d agcara pemer...,a i s e n o d n I k i l b u p e a R i s g e n ...,direktori putusan mahkamah agung republik indo...,0.190215
20,case_021,600/Pid.B/2021/PN,Pengadilan Negeri Cibinong yang mengadili perk...,2021,Irfan Saifudin 2. Tempat lahir : Jakarta I h3....,-,mengadili perkara pidana doengan u acara pemer...,a i s e n o d n I k i l b u p e a R i s g e n ...,direktori putusan mahkamah agung republik indo...,0.188970
5,case_006,240/Pid.B/2022/PN,Pengadilan Negeri Cibinong yang mengadili perk...,2022,Rizki Aditia Ginting als Adit Bin Manus Gintin...,-,mengadili perkara pidana dengan Aacara pemerik...,a i s e n o d n I k i l b u p e a R i s g e n ...,direktori putusan mahkamah agung republik indo...,0.184441
8,case_009,284/Pid.B/2020/PN,Pengadilan Negeri Cibinong yang mengadili perk...,2020,Ibrohim Alias Ibro Bin Parta (Alm)I h2. Tempat...,-,mengadili perkara pidana doengan u acara pemer...,a i s e n o d n I k i l b u p e a R i s g e n ...,direktori putusan mahkamah agung republik indo...,0.182546
14,case_015,387/Pid.B/2022/PN,Pengadilan Negeri Cibinong yang mengadili perk...,2022,Afriansyah Bin Samhudi I h2. Tempat lahir : Ta...,-,mengadili perkara pidana doengan u acara pemer...,a i s e n o d n I k i l b u p e a R i s g e n ...,direktori putusan mahkamah agung republik indo...,0.178744


In [51]:
print("="*50)

print("Jumlah Case :",len(df))

print("Similarity Max :",similarity.max())

print("Similarity Min :",similarity.min())

print("Similarity Mean :",similarity.mean())

Jumlah Case : 30
Similarity Max : 0.9557026160394054
Similarity Min : 0.06849630689466621
Similarity Mean : 0.20081808555502284
